DeepRAG enhances LLMs by grounding their responses in external knowledge. The workflow has four main stages:
- Document Ingestion
- Load raw data (PDFs, text, CSVs, web pages).
- Normalize and clean content.
- Chunking & Embedding
- Split documents into overlapping chunks (e.g., 500 tokens with 50 overlap).
- Convert chunks into embeddings using OpenAI’s text-embedding-ada-002.
- Vector Store & Retrieval
- Store embeddings in FAISS, Pinecone, or Weaviate.
- Query vector store with user input → retrieve top‑k relevant chunks.
- LLM Generation
- Pass retrieved context + query into GPT (e.g., gpt-4).
- Generate grounded, context‑aware answers.


# ============================
# 1. Install Dependencies
# ============================

In [1]:
!pip install openai faiss-cpu langchain pypdf pandas scikit-learn

# ============================
# 2. Import Libraries
# ============================

In [2]:
import openai
import pandas as pd
from langchain.vectorstores import FAISS
from langchain.llms import OpenAI
from langchain.document_loaders import PyPDFLoader
from sklearn.metrics import precision_score, recall_score
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.document_loaders import PyPDFLoader

# ============================
# 3. Load PDF Documents
# ============================

In [3]:
pdf_path = "HR_Policy_Manual_2023.pdf"   # Replace with your PDF file
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# ============================
# 4. Automatic Chunking
# ============================

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(documents)

# ============================
# 5. Create Embeddings & Vector Store
# ============================

In [5]:
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
vectorstore = FAISS.from_documents(docs, embeddings)

# ============================
# 7. Build RAG Pipeline
# ============================

In [6]:
llm = ChatOpenAI(model="gpt-4", temperature=0)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",   # simple chain type
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

# ============================
# 8. Query the System
# ============================

In [7]:
query = "Summarize the main topic of this PDF."
response = qa.invoke(query)
print("Q:", query)
print("A:", response)

Q: Summarize the main topic of this PDF.
A: {'query': 'Summarize the main topic of this PDF.', 'result': 'The main topic of this PDF is the HR Policy Manual for IIMA (Indian Institute of Management Ahmedabad) for the year 2023. It covers various policies and procedures related to human resources at IIMA. The specific chapter mentioned, Chapter 23, deals with the retention of documents.', 'source_documents': [Document(page_content='Summary ................................................................................................................................................ 164', metadata={'source': 'HR_Policy_Manual_2023.pdf', 'page': 8}), Document(page_content='Signature:  ____________________________________\nName       :  __________________________________\nDate      :  ________________________________', metadata={'source': 'HR_Policy_Manual_2023.pdf', 'page': 49}), Document(page_content='iii\nIIMA HR Policy Manual 2023\nCONTENTS\nContents\nChapter 1 General ................

In [8]:
# ============================
# 9. Evaluation Matrix
# ============================
# Define test queries and expected answers (gold standard)

In [9]:
test_data = [
    {"query": "How many leave employee can take?", "expected": "Retrieval-Augmented Generation"},
    {"query": "How many CL can take?", "expected": "OpenAI embeddings"},
    {"query": "Which database stores vectors?", "expected": "FAISS"}
]

results = []
for item in test_data:
    ans = qa.invoke(item["query"])
    results.append({"query": item["query"], "expected": item["expected"], "answer": ans})

df = pd.DataFrame(results)
print(df)

# Simple evaluation: check if expected keywords appear in answers
df["match"] = df.apply(lambda row: row["expected"]  in row["answer"] , axis=1)

precision = precision_score(df["match"], [True]*len(df), zero_division=0)
recall = recall_score(df["match"], [True]*len(df), zero_division=0)

print("Precision:", precision)
print("Recall:", recall)

                               query                        expected  \
0  How many leave employee can take?  Retrieval-Augmented Generation   
1              How many CL can take?               OpenAI embeddings   
2     Which database stores vectors?                           FAISS   

                                              answer  
0  {'query': 'How many leave employee can take?',...  
1  {'query': 'How many CL can take?', 'result': '...  
2  {'query': 'Which database stores vectors?', 'r...  
Precision: 0.0
Recall: 0.0
